In [4]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from tdc.benchmark_group import admet_group

In [5]:
BENCHMARK_NAME = "Solubility_AqSolDB"

PROCESSED_DIR = Path("../data/processed/aqsoldb")

RANDOM_STATE = 42

In [6]:
feature_sets = {
    "descriptorsRO5": pd.read_parquet(
        PROCESSED_DIR / "train_val_descriptors_RO5.parquet"
    ),

    "descriptorsESOL": pd.read_parquet(
        PROCESSED_DIR / "train_val_descriptors_ESOL.parquet"
    ),

    "descriptorsEXT": pd.read_parquet(
        PROCESSED_DIR / "train_val_descriptors_extended.parquet"
    ),

    "maccs": pd.read_parquet(
        PROCESSED_DIR / "train_val_maccs.parquet"
    ),

    "morgan": pd.read_parquet(
        PROCESSED_DIR / "train_val_morgan.parquet"
    ),

    "descRO5_maccs": pd.read_parquet(
        PROCESSED_DIR / "train_val_descRO5_maccs.parquet"
    ),

    "descESOL_maccs": pd.read_parquet(
        PROCESSED_DIR / "train_val_descESOL_maccs.parquet"
    ),

    "descEXT_maccs": pd.read_parquet(
        PROCESSED_DIR / "train_val_descEXT_maccs.parquet"
    ),

    "descRO5_morgan": pd.read_parquet(
        PROCESSED_DIR / "train_val_descRO5_morgan.parquet"
    ),

    "descESOL_morgan": pd.read_parquet(
        PROCESSED_DIR / "train_val_descESOL_morgan.parquet"
    ),

    "descEXT_morgan": pd.read_parquet(
        PROCESSED_DIR / "train_val_descEXT_morgan.parquet"
    ),
}

In [7]:
y = pd.read_parquet(
    PROCESSED_DIR / "train_val_target.parquet"
)["Y"]

In [8]:
metadata = pd.read_csv(
    PROCESSED_DIR / "train_val_metadata.csv"
)

In [9]:
group = admet_group(path="../data/raw/")

train_split, valid_split = group.get_train_valid_split(
    benchmark=BENCHMARK_NAME,
    split_type="default",
    seed=RANDOM_STATE,
)

Found local copy...
generating training, validation splits...
100%|██████████| 7985/7985 [00:01<00:00, 5972.81it/s] 


In [10]:
train_drugs = set(train_split["Drug"])
valid_drugs = set(valid_split["Drug"])

train_mask = metadata["Drug"].isin(train_drugs)
valid_mask = metadata["Drug"].isin(valid_drugs)

In [11]:
print("Train:", train_mask.sum())
print("Validation:", valid_mask.sum())

assert (train_mask & valid_mask).sum() == 0
assert train_mask.sum() + valid_mask.sum() == len(metadata)

Train: 5045
Validation: 2940


In [12]:
y_train = y.loc[train_mask].reset_index(drop=True)
y_valid = y.loc[valid_mask].reset_index(drop=True)

In [13]:
def evaluate_regression(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(
            y_true,
            y_pred,
        ),

        "RMSE": mean_squared_error(
            y_true,
            y_pred,
        ) ** 0.5,

        "R2": r2_score(
            y_true,
            y_pred,
        ),
    }

In [14]:
models = {
    "dummy": DummyRegressor(
        strategy="mean"
    ),

    "ridge": Pipeline([
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "model",
            Ridge(alpha=1.0),
        ),
    ]),

    "random_forest": RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),

    "gradient_boosting": GradientBoostingRegressor(
        random_state=RANDOM_STATE,
    ),
}

In [15]:
results = []

for feature_name, X in feature_sets.items():

    X_train = X.loc[
        train_mask
    ].reset_index(drop=True)

    X_valid = X.loc[
        valid_mask
    ].reset_index(drop=True)

    for model_name, model in models.items():

        model.fit(
            X_train,
            y_train,
        )

        y_pred = model.predict(
            X_valid
        )

        metrics = evaluate_regression(
            y_valid,
            y_pred,
        )

        results.append({
            "features": feature_name,
            "model": model_name,
            **metrics,
        })

In [16]:
results_df = pd.DataFrame(results)

results_df.sort_values("MAE")

,features,model,MAE,RMSE,R2
31,descEXT_maccs,gradient_boosting,1.055431,1.714265,0.493563
27,descESOL_maccs,gradient_boosting,1.059989,1.684055,0.511256
23,descRO5_maccs,gradient_boosting,1.070639,1.736131,0.480562
30,descEXT_maccs,random_forest,1.085072,1.724348,0.487588
26,descESOL_maccs,random_forest,1.086637,1.742426,0.476788
22,descRO5_maccs,random_forest,1.091493,1.756071,0.468561
35,descRO5_morgan,gradient_boosting,1.098394,1.709019,0.496658
38,descESOL_morgan,random_forest,1.101003,1.788073,0.449015
42,descEXT_morgan,random_forest,1.101299,1.782071,0.452708
43,descEXT_morgan,gradient_boosting,1.103748,1.723799,0.487914


In [17]:
FEATURES_TO_EVALUATE = [
    "descRO5_maccs",
    "descESOL_maccs",
    "descEXT_maccs",
]

SEEDS = [0, 1, 2, 3, 4]

MULTI_SEED_MODELS = {
    "gradient_boosting": GradientBoostingRegressor(
        random_state=RANDOM_STATE,
    ),

    "random_forest": RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

multi_seed_results = []
prediction_results = []

for seed in SEEDS:

    # Generate the TDC scaffold split for this seed
    train_split, valid_split = group.get_train_valid_split(
        benchmark=BENCHMARK_NAME,
        split_type="default",
        seed=seed,
    )

    train_drugs = set(train_split["Drug"])
    valid_drugs = set(valid_split["Drug"])

    train_mask = metadata["Drug"].isin(train_drugs)
    valid_mask = metadata["Drug"].isin(valid_drugs)

    assert not (train_mask & valid_mask).any()
    assert train_mask.sum() + valid_mask.sum() == len(metadata)

    y_train = (
        y.loc[train_mask]
        .reset_index(drop=True)
    )

    y_valid = (
        y.loc[valid_mask]
        .reset_index(drop=True)
    )

    valid_metadata = (
        metadata.loc[valid_mask]
        .reset_index(drop=True)
    )

    for feature_name in FEATURES_TO_EVALUATE:

        X = feature_sets[feature_name]

        X_train = (
            X.loc[train_mask]
            .reset_index(drop=True)
        )

        X_valid = (
            X.loc[valid_mask]
            .reset_index(drop=True)
        )

        for model_name, model in MULTI_SEED_MODELS.items():

            model.fit(
                X_train,
                y_train,
            )

            y_pred = model.predict(X_valid)

            metrics = evaluate_regression(
                y_valid,
                y_pred,
            )

            # Experiment-level metrics
            multi_seed_results.append({
                "seed": seed,
                "features": feature_name,
                "model": model_name,
                **metrics,
            })

            # Molecule-level predictions
            seed_predictions = pd.DataFrame({
                "seed": seed,
                "Drug": valid_metadata["Drug"],
                "features": feature_name,
                "model": model_name,
                "y_true": y_valid,
                "y_pred": y_pred,
            })

            prediction_results.append(
                seed_predictions
            )

generating training, validation splits...
100%|██████████| 7985/7985 [00:01<00:00, 6400.88it/s] 
generating training, validation splits...
100%|██████████| 7985/7985 [00:01<00:00, 6412.87it/s] 
generating training, validation splits...
100%|██████████| 7985/7985 [00:01<00:00, 6611.06it/s] 
generating training, validation splits...
100%|██████████| 7985/7985 [00:01<00:00, 6451.40it/s] 
generating training, validation splits...
100%|██████████| 7985/7985 [00:01<00:00, 6905.26it/s] 


In [18]:
validation_metrics = pd.DataFrame(
    multi_seed_results
)

validation_predictions = pd.concat(
    prediction_results,
    ignore_index=True,
)

In [29]:
RESULTS_DIR = Path("../data/results/aqsoldb")
RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [30]:
validation_metrics.to_csv(
    RESULTS_DIR / "validation_metrics.csv",
    index=False,
)

In [31]:
validation_predictions.to_csv(
    RESULTS_DIR / "validation_predictions.csv",
    index=False,
)